In [ ]:
%run -i ../../python_scripts/nb_setup.py

### Diabetes 30-day readmission 

In [2]:
LOCAL_CSV = "diabetic_data.csv"  # if absent, fetched via ucimlrepo
TOP_LEVELS = 50
SEED, OUT = 0, "sgp_set_tabular_SR"
DEDUP = False
GROUP_SPLIT = True
FRACS = (0.55, 0.70)
N_SEEDS = 5  # predictions averaged over seeds

In [3]:
if os.path.exists(LOCAL_CSV):
    df_raw = pd.read_csv(LOCAL_CSV, na_values="?")
else:
    from ucimlrepo import fetch_ucirepo

    r = fetch_ucirepo(id=296)
    print(r.metadata.name)
    parts = [getattr(r.data, "ids", None), r.data.features, r.data.targets]
    df_raw = pd.concat([x for x in parts if x is not None], axis=1)

if "discharge_disposition_id" in df_raw:
    df_raw = df_raw[~df_raw.discharge_disposition_id.isin([11, 13, 14, 19, 20, 21])]
df_raw = df_raw.reset_index(drop=True)
print(df_raw.shape)

Diabetes 130-US Hospitals for Years 1999-2008
(99343, 50)


In [4]:
def icd9_group(code):
    """Strack et al. (2014) grouping: 700+ ICD-9 codes -> 10 clinical buckets."""
    s = str(code).strip()
    if s in ("nan", "?", "", "None"):
        return "missing"
    if s[0] == "V":
        return "other"
    if s[0] == "E":
        return "injury"
    try:
        i = int(float(s))
    except ValueError:
        return "other"
    if i == 250:
        return "diabetes"
    if 390 <= i <= 459 or i == 785:
        return "circulatory"
    if 460 <= i <= 519 or i == 786:
        return "respiratory"
    if 520 <= i <= 579 or i == 787:
        return "digestive"
    if 800 <= i <= 999:
        return "injury"
    if 710 <= i <= 739:
        return "musculoskeletal"
    if 580 <= i <= 629 or i == 788:
        return "genitourinary"
    if 140 <= i <= 239:
        return "neoplasms"
    return "other"


def build(df, dedup=DEDUP, top_levels=TOP_LEVELS):
    if dedup and "patient_nbr" in df:
        df = df.drop_duplicates("patient_nbr")
    df = df.reset_index(drop=True)

    y = (df.readmitted.astype(str).str.strip() == "<30").to_numpy().astype(int)
    pid = pd.factorize(df.patient_nbr)[0] if "patient_nbr" in df else np.arange(len(df))

    drop = [
        c for c in ["encounter_id", "patient_nbr", "weight", "readmitted"] if c in df
    ]
    X = df.drop(columns=drop).copy()

    # the 23 medication columns take values in {No, Steady, Up, Down}
    drugs = [
        c
        for c in X.columns
        if X[c].dtype == object
        and set(X[c].dropna().unique()) <= {"No", "Steady", "Up", "Down"}
    ]

    X["service_utilization"] = (
        df.number_outpatient + df.number_emergency + df.number_inpatient
    )
    X["n_meds_on"] = (df[drugs] != "No").sum(axis=1)
    X["n_med_changes"] = df[drugs].isin(["Up", "Down"]).sum(axis=1)
    X["age_mid"] = df.age.map(
        {f"[{10 * i}-{10 * i + 10})": 10 * i + 5 for i in range(10)}
    )

    for c in ["diag_1", "diag_2", "diag_3"]:
        if c in X:
            X[c] = df[c].map(icd9_group)

    # nominal codes stored as ints -> trees were splitting them as if ordered
    for c in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
        if c in X:
            X[c] = X[c].astype(str)

    for c in X.select_dtypes("object"):
        top = X[c].value_counts().index[:top_levels]
        # NaN kept as NaN: HistGB handles missing values
        X[c] = X[c].where(X[c].isin(top) | X[c].isna(), "other").astype("category")

    return X, y, pid


X, y, pid = build(df_raw)
print(X.shape, "| patients:", pid.max() + 1, "| positives:", round(y.mean(), 4))

(99343, 50) | patients: 69990 | positives: 0.1139


In [5]:
# Split & fit
PARAMS = dict(
    categorical_features="from_dtype",  # requires scikit-learn >= 1.4
    learning_rate=0.03,
    max_iter=2000,  # early stopping decides the real depth
    max_leaf_nodes=31,
    min_samples_leaf=100,
    l2_regularization=1.0,
    max_features=0.8,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=40,
    scoring="roc_auc",  # the metric we actually care about
)


def split_masks(pid, group=GROUP_SPLIT, fracs=FRACS, seed=SEED):
    key = pid if group else np.arange(len(pid))
    u = np.random.default_rng(seed).random(key.max() + 1)[key]
    a, b = fracs
    return u < a, (u >= a) & (u < b), u >= b


def fit_predict(X, y, fit, n_seeds=N_SEEDS, verbose=True):
    ps = []
    for s in range(n_seeds):
        m = HistGradientBoostingClassifier(random_state=SEED + s, **PARAMS)
        m.fit(X[fit], y[fit])
        ps.append(m.predict_proba(X)[:, 1])
        if verbose:
            print(f"  seed {s}: {m.n_iter_} iters")
    return np.mean(ps, axis=0)


fit, tau_split, sn = split_masks(pid)
print("fit/tau/Sn =", fit.sum(), tau_split.sum(), sn.sum())

p = fit_predict(X, y, fit)
print("AUC (Sn) =", round(roc_auc_score(y[sn], p[sn]), 3))

fit/tau/Sn = 54685 14774 29884
  seed 0: 205 iters
  seed 1: 158 iters
  seed 2: 152 iters
  seed 3: 233 iters
  seed 4: 247 iters
AUC (Sn) = 0.68


In [6]:
# Threshold fitted on its own slice (Youden's J), then absorbed into the head's bias
fpr, tpr, thr = roc_curve(y[tau_split], p[tau_split])
tau = float(np.clip(thr[np.argmax(tpr - fpr)], 1e-6, 1 - 1e-6))

s = logit(np.clip(p, 1e-6, 1 - 1e-6)) - logit(tau)
p_shift = expit(s)
sgp_df = pd.DataFrame(
    {
        "y_true": y.astype(float),
        "y_pred": (s >= 0).astype(float),
        "kappa": np.maximum(p_shift, 1 - p_shift),  # softmax response
    }
)[sn].reset_index(drop=True)
print("tau =", round(tau, 4))

tau = 0.1326


In [7]:
def report(df, name):
    n = len(df)
    err = (df.y_pred != df.y_true).mean()
    fp = ((df.y_pred == 1) & (df.y_true == 0)).sum()
    fn = ((df.y_pred == 0) & (df.y_true == 1)).sum()
    ppv = ((df.y_pred == 1) & (df.y_true == 1)).sum() / max((df.y_pred == 1).sum(), 1)
    print(
        f"{name}: N={n} | positives={df.y_true.mean():.3f} | 0/1 risk={err:.3f} | "
        f"FP={fp} FN={fn} | FPR={fp/(df.y_true==0).sum():.3f} "
        f"FNR={fn/(df.y_true==1).sum():.3f} | PPV={ppv:.3f} | "
        f"kappa in [{df.kappa.min():.3f}, {df.kappa.max():.3f}]"
    )


report(sgp_df, "sgp_set")
pickle.dump(sgp_df, open(OUT, "wb"))
sgp_df.head(3)

sgp_set: N=29884 | positives=0.116 | 0/1 risk=0.275 | FP=6459 FN=1750 | FPR=0.245 FNR=0.503 | PPV=0.211 | kappa in [0.500, 0.945]


,y_true,y_pred,kappa
0,0.0,0.0,0.655985
1,0.0,0.0,0.644825
2,0.0,0.0,0.613991
